# PWV03 : Two-Point Temporal Correlation Function

**Goal :** Characterise the temporal coherence of PWV at Cerro Pachon within
a single observing night — from a few minutes to 10 hours.

The **two-point temporal correlation function** is defined as:
$$
C(\Delta t) = \frac{\langle \delta\mathrm{PWV}(t)\,\delta\mathrm{PWV}(t+\Delta t)\rangle}
                   {\langle \delta\mathrm{PWV}^2 \rangle}
\qquad \text{with} \quad \delta\mathrm{PWV}(t) = \mathrm{PWV}(t) - \langle\mathrm{PWV}\rangle
$$

By construction $C(0)=1$ and $C(\Delta t)\to 0$ for uncorrelated separations.

The lag range is restricted to **10 hours** (600 min) — the typical length of
an AuxTel observing night — so that all pairs remain within the same night
and the correlation reflects genuine atmospheric coherence rather than
seasonal or inter-night trends.

Two complementary views:
- **Linear x-axis in hours** (0 – 10 h): intuitive view of the decorrelation.
- **Logarithmic x-axis in minutes** (1 min – 600 min): equal weight per
  decade, reveals the short-timescale power-law behaviour.

**Author :** Sylvie Dagoret-Campagne — IJCLab/IN2P3/CNRS  
**Created :** 2026-03-19  
**Kernel :** conda_py313

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from astropy.time import Time

%matplotlib inline

# ── publication rc params ─────────────────────────────────────────────────────
mpl.rcParams.update({
    'figure.dpi'     : 150,
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.labelsize' : 13,
    'axes.titlesize' : 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'axes.linewidth' : 1.1,
})

In [ ]:
from PWV00_parameters import (
    version_run, tag, extractedfilesdict,
    PWV_FILTER_LIST, FLAG_PWVFILTERS,
    DumpConfig
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter

DumpConfig()

In [ ]:
pathfigs = "figs_PWV03corr"
prefix   = "pwv03corr"
os.makedirs(pathfigs, exist_ok=True)
figtype  = ".pdf"

---
## 1. Load data & build Time column

In [ ]:
atmfilename   = extractedfilesdict[version_run]
inputfilename = atmfilename.split("/")[-1]
print(f"Loading: {atmfilename}")

if "parquet" in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec  = pd.DataFrame(specdata)

# Time from DATE-OBS (same as PWV01/02)
df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"], utc=True)
df_spec = df_spec.sort_values("Time", ascending=True).reset_index(drop=True)

print(f"Shape: {df_spec.shape}")
print(f"Date range: {df_spec['Time'].min().date()}  ->  {df_spec['Time'].max().date()}")

In [ ]:
if "run2026_v02" in version_run:
    df_spec["chi2_ram"] = df_spec["CHI2_FIT"]
    df_spec["chi2_rum"] = df_spec["CHI2_FIT"]

if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec["FILTER"].isin(PWV_FILTER_LIST)].copy()

pwv_cols = ["PWV [mm]_ram", "PWV [mm]_rum", "PWV [mm]_err_ram", "PWV [mm]_err_rum"]
df_spec.dropna(subset=pwv_cols, inplace=True)

print(f"After filter selection: {len(df_spec)} rows")

In [ ]:
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="CHI2_FIT", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_ram", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_rum", ext="norm")

---
## 2. Quality cuts (tight)

In [ ]:
pathdata      = "data_PWV01seas"
filename_cuts = f"{pathdata}/cuts_tight_finaldecision.json"

cuts           = ParameterCutTools.load_cuts_json(filename_cuts)
list_of_params = list(cuts.keys())

selector    = ParameterCutSelection(df_spec, params=list_of_params, id_col="id")
flags       = selector.apply_cuts(cuts)
df_selected = df_spec.merge(flags, on="id")
df_keep     = df_selected[df_selected["pass_all_cuts"]].copy()

print(f"Tight cuts: {len(df_keep)} / {len(df_spec)} kept")

---
## 3. Prepare PWV time series

Use the mean of the two estimators (ram, rum) as the best PWV value.
The MJD column gives absolute time in days — we work in **minutes** internally
for all lag computations.

In [ ]:
df_keep["PWV"]     = 0.5 * (df_keep["PWV [mm]_ram"] + df_keep["PWV [mm]_rum"])
df_keep["PWV_err"] = np.sqrt(
    df_keep["PWV [mm]_err_ram"]**2 + df_keep["PWV [mm]_err_rum"]**2
)

# Time in minutes since first observation (float64)
t0_min = df_keep["MJD"].min() * 1440.0          # MJD -> minutes
df_keep["t_min"] = df_keep["MJD"] * 1440.0 - t0_min

# Mean-subtracted PWV (fluctuation field)
pwv_mean          = df_keep["PWV"].mean()
df_keep["dPWV"]   = df_keep["PWV"] - pwv_mean
pwv_var           = df_keep["dPWV"].var(ddof=1)   # variance = C(0) denominator

print(f"N observations   : {len(df_keep)}")
print(f"<PWV>            : {pwv_mean:.3f} mm")
print(f"Var(PWV)         : {pwv_var:.4f} mm^2   => std = {np.sqrt(pwv_var):.3f} mm")
print(f"Time span        : {df_keep['t_min'].max()/1440:.1f} days  "
      f"({df_keep['t_min'].max()/60:.0f} hours)")

---
## 4. Two-point correlation — binned estimator

Because the data are **irregularly sampled**, we compute the correlation
by forming all pairs $(i, j)$ with $i < j$ and $\delta t_{ij} \leq 10\,\mathrm{h}$,
then averaging in logarithmic lag bins.

$$
C(\Delta t_k) = \frac{1}{N_k\,\sigma^2}\sum_{\{i,j\}\in\mathrm{bin}_k}
                \delta\mathrm{PWV}_i\,\delta\mathrm{PWV}_j
$$

Restricting to $\Delta t \leq 600\,\mathrm{min}$ (one night) ensures
all pairs are intra-night and the correlation reflects genuine atmospheric
coherence rather than seasonal modulation.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
LAG_MIN_MIN   =   1.0    # minimum lag: 1 minute
LAG_MAX_MIN   = 600.0    # maximum lag: 10 hours = 600 minutes (within one night)
N_BINS_LOG    =  50      # number of log-spaced bins over [1 min, 600 min]

# Sub-sampling: max number of points for the all-pairs computation
# With N=2000 and pairs filtered to dt<=600 min, memory is fine
N_MAX_SUBSAMPLE = 2000

rng = np.random.default_rng(seed=42)

t_arr  = df_keep["t_min"].values   # minutes
dp_arr = df_keep["dPWV"].values    # mm, mean-subtracted

N_obs = len(t_arr)
if N_obs > N_MAX_SUBSAMPLE:
    idx    = rng.choice(N_obs, size=N_MAX_SUBSAMPLE, replace=False)
    idx    = np.sort(idx)
    t_s    = t_arr[idx]
    dp_s   = dp_arr[idx]
    print(f"Sub-sampling {N_MAX_SUBSAMPLE} / {N_obs} points for pair computation")
else:
    t_s, dp_s = t_arr, dp_arr
    print(f"Using all {N_obs} points")

N_s = len(t_s)
print(f"Lag range        : {LAG_MIN_MIN:.0f} min – {LAG_MAX_MIN:.0f} min ({LAG_MAX_MIN/60:.0f} h)")

In [ ]:
# ── Compute all pairs within [LAG_MIN_MIN, LAG_MAX_MIN] ──────────────────────
i_idx, j_idx = np.triu_indices(N_s, k=1)

dt_all    = t_s[j_idx]  - t_s[i_idx]   # lag in minutes (>0 by construction)
prod_all  = dp_s[i_idx] * dp_s[j_idx]

# Keep only pairs within the 10-hour window
in_range  = (dt_all >= LAG_MIN_MIN) & (dt_all <= LAG_MAX_MIN)
dt_pairs  = dt_all[in_range]
prod_pairs = prod_all[in_range]

print(f"Total pairs (all lags) : {len(dt_all):,}")
print(f"Pairs within 10 h      : {len(dt_pairs):,}")
print(f"Lag range (kept)       : {dt_pairs.min():.2f} – {dt_pairs.max():.1f} min")

In [ ]:
# ── Log-spaced lag bins over [1 min, 600 min] ─────────────────────────────────
bin_edges = np.logspace(
    np.log10(LAG_MIN_MIN),
    np.log10(LAG_MAX_MIN),
    N_BINS_LOG + 1
)  # minutes

bin_centers_min = np.sqrt(bin_edges[:-1] * bin_edges[1:])  # geometric mean

bin_idx = np.digitize(dt_pairs, bin_edges) - 1
valid   = (bin_idx >= 0) & (bin_idx < N_BINS_LOG)

corr     = np.full(N_BINS_LOG, np.nan)
n_pairs  = np.zeros(N_BINS_LOG, dtype=int)
corr_err = np.full(N_BINS_LOG, np.nan)

for k in range(N_BINS_LOG):
    mask = valid & (bin_idx == k)
    nk   = mask.sum()
    n_pairs[k] = nk
    if nk == 0:
        continue
    prods_k    = prod_pairs[mask]
    corr[k]    = prods_k.mean() / pwv_var
    corr_err[k] = prods_k.std(ddof=1) / (np.sqrt(nk) * pwv_var)

lag_min = bin_centers_min          # minutes
lag_hr  = bin_centers_min / 60.0   # hours

MIN_PAIRS = 5
good = n_pairs >= MIN_PAIRS
print(f"Bins with >= {MIN_PAIRS} pairs: {good.sum()} / {N_BINS_LOG}")

---
## 5. Decorrelation timescale

In [ ]:
df_corr = pd.DataFrame({
    "lag_min" : lag_min[good],
    "lag_hr"  : lag_hr[good],
    "C"       : corr[good],
    "C_err"   : corr_err[good],
    "n_pairs" : n_pairs[good],
})

# 1/e crossing
inv_e = 1.0 / np.e
cross = df_corr[df_corr["C"] < inv_e]
if len(cross) > 0:
    tau_min = cross.iloc[0]["lag_min"]
    tau_hr  = cross.iloc[0]["lag_hr"]
    print(f"Decorrelation timescale (C < 1/e) : {tau_min:.1f} min = {tau_hr:.2f} h")
else:
    tau_min = np.nan
    tau_hr  = np.nan
    print("C(Delta_t) does not drop below 1/e within 10 h")

display(df_corr)

---
## 6. Main figure — 2-panel  (linear hours | log minutes)

Left panel : x-axis = lag in **hours**, **linear** scale, range 0 – 10 h  
Right panel : x-axis = lag in **minutes**, **log** scale, range 1 – 600 min  
Both y-axes : $C(\Delta t)$, linear scale.

In [ ]:
CORR_COLOR  = "steelblue"
ERR_COLOR   = "steelblue"
ZERO_COLOR  = "gray"
INV_E_COLOR = "darkorange"
TAU_COLOR   = "darkred"


def _decorate_corr_ax(ax, x_tau=None, xlabel="", xscale="linear",
                      xlim=None, ylim=(-0.25, 1.05)):
    """Common decorations for a correlation panel."""
    ax.axhline(0.0,      ls="--", color=ZERO_COLOR,  lw=1.0, zorder=1)
    ax.axhline(1.0/np.e, ls=":",  color=INV_E_COLOR, lw=1.5, zorder=1,
               label=r"$C = 1/e$")
    if x_tau is not None and np.isfinite(x_tau):
        unit = xlabel.split('[')[1].rstrip(']') if '[' in xlabel else ''
        ax.axvline(x_tau, ls="-", color=TAU_COLOR, lw=1.5, zorder=2,
                   label=rf"$\tau_{{1/e}} = {x_tau:.1f}$ {unit}")
    ax.set_ylabel(r"$C(\Delta t)$", fontsize=13)
    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_xscale(xscale)
    if xlim is not None:
        ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.legend(fontsize=9, loc="upper right")


fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.subplots_adjust(wspace=0.32)

# ── LEFT : linear x in hours, 0 – 10 h ────────────────────────────────────────
ax_lin = axes[0]
ax_lin.fill_between(
    lag_hr[good], corr[good] - corr_err[good], corr[good] + corr_err[good],
    alpha=0.25, color=ERR_COLOR
)
ax_lin.plot(lag_hr[good], corr[good],
            color=CORR_COLOR, lw=1.8, marker="o", ms=3.5, zorder=3,
            label=r"$C(\Delta t)$")
_decorate_corr_ax(
    ax_lin,
    x_tau  = tau_hr if np.isfinite(tau_hr) else None,
    xlabel = r"$\Delta t$ [hours]",
    xscale = "linear",
    xlim   = (0, LAG_MAX_MIN / 60.0),   # 0 – 10 h
)
ax_lin.set_title("Two-point temporal correlation — linear hours", fontsize=11)

# ── RIGHT : log x in minutes, 1 – 600 min ─────────────────────────────────────
ax_log = axes[1]
ax_log.fill_between(
    lag_min[good], corr[good] - corr_err[good], corr[good] + corr_err[good],
    alpha=0.25, color=ERR_COLOR
)
ax_log.plot(lag_min[good], corr[good],
            color=CORR_COLOR, lw=1.8, marker="o", ms=3.5, zorder=3,
            label=r"$C(\Delta t)$")
_decorate_corr_ax(
    ax_log,
    x_tau  = tau_min if np.isfinite(tau_min) else None,
    xlabel = r"$\Delta t$ [minutes]",
    xscale = "log",
    xlim   = (LAG_MIN_MIN, LAG_MAX_MIN),   # 1 – 600 min
)
ax_log.set_title("Two-point temporal correlation — log minutes", fontsize=11)

# Secondary top axis: hours on the log panel
ax_log2 = ax_log.twiny()
ax_log2.set_xlim(np.array(ax_log.get_xlim()) / 60.0)
ax_log2.set_xscale("log")
ax_log2.set_xlabel(r"$\Delta t$ [hours]", fontsize=10, labelpad=6)

fig.suptitle(
    f"PWV two-point temporal correlation (0–10 h) — {version_run} (tight cuts)",
    fontsize=12, y=1.01
)

figfile1 = f"{pathfigs}/{prefix}_{version_run}_2point_corr_2panel{figtype}"
fig.savefig(figfile1, bbox_inches="tight")
print(f"Saved: {figfile1}")
plt.show()

---
## 7. Zoom: very short timescales (log minutes, < 60 min)

Highlights sub-hour intra-night decorrelation.

In [ ]:
ZOOM_MAX_MIN = 60.0   # 1 hour in minutes
short = good & (lag_min <= ZOOM_MAX_MIN)

fig2, ax2 = plt.subplots(figsize=(10, 5))

ax2.fill_between(
    lag_min[short], corr[short] - corr_err[short], corr[short] + corr_err[short],
    alpha=0.25, color=ERR_COLOR
)
ax2.plot(lag_min[short], corr[short],
         color=CORR_COLOR, lw=2, marker="o", ms=4.5, zorder=3,
         label=r"$C(\Delta t)$")
_decorate_corr_ax(
    ax2,
    x_tau  = tau_min if (np.isfinite(tau_min) and tau_min <= ZOOM_MAX_MIN) else None,
    xlabel = r"$\Delta t$ [minutes]",
    xscale = "log",
    xlim   = (LAG_MIN_MIN, ZOOM_MAX_MIN),
)
ax2.set_title(
    r"Short-timescale PWV correlation ($\Delta t < 1$ h, log scale)",
    fontsize=11
)

# secondary axis in hours
ax2b = ax2.twiny()
ax2b.set_xlim(np.array(ax2.get_xlim()) / 60.0)
ax2b.set_xscale("log")
ax2b.set_xlabel(r"$\Delta t$ [hours]", fontsize=10, labelpad=6)

fig2.suptitle(
    f"PWV sub-hour correlation — {version_run} (tight cuts)",
    fontsize=12, y=1.02
)

figfile2 = f"{pathfigs}/{prefix}_{version_run}_2point_corr_subhour{figtype}"
fig2.savefig(figfile2, bbox_inches="tight")
print(f"Saved: {figfile2}")
plt.show()

---
## 8. Number of pairs per bin

In [ ]:
fig3, ax3 = plt.subplots(figsize=(12, 4))

ax3.bar(
    np.arange(N_BINS_LOG)[good],
    n_pairs[good],
    width=0.8,
    color="steelblue", alpha=0.8, edgecolor="white", linewidth=0.4
)
ax3.set_yscale("log")
ax3.set_xlabel("Bin index", fontsize=12)
ax3.set_ylabel("Number of pairs", fontsize=12)
ax3.set_title("Pair count per log-lag bin  (1 min – 10 h)", fontsize=11)

tick_idx = np.linspace(0, N_BINS_LOG - 1, 8, dtype=int)
ax3.set_xticks(tick_idx)
ax3.set_xticklabels(
    [f"{lag_min[k]:.1f} min" if lag_min[k] < 60
     else f"{lag_hr[k]:.1f} h"
     for k in tick_idx],
    rotation=30, ha="right", fontsize=9
)

fig3.tight_layout()
figfile3 = f"{pathfigs}/{prefix}_{version_run}_pair_counts{figtype}"
fig3.savefig(figfile3, bbox_inches="tight")
print(f"Saved: {figfile3}")
plt.show()

---
## 9. Separate correlations per filter

Check whether different filters (empty, OG550, FELH0600) yield consistent
temporal correlation structures within the 10-hour window.

In [ ]:
FILTERS_OF_INTEREST = ["empty", "OG550_65mm_1", "FELH0600"]
filter_color = {
    "empty"        : "#1f77b4",
    "OG550_65mm_1" : "#d62728",
    "FELH0600"     : "#2ca02c",
}
filter_label = {
    "empty"        : "empty",
    "OG550_65mm_1" : "OG550",
    "FELH0600"     : "FELH0600",
}

corr_by_filter     = {}
corr_err_by_filter = {}
npairs_by_filter   = {}

for filt in FILTERS_OF_INTEREST:
    sub = df_keep[df_keep["FILTER"] == filt]
    if len(sub) < 10:
        print(f"  {filt}: too few points ({len(sub)}), skipped")
        continue

    t_f   = sub["t_min"].values
    dp_f  = sub["dPWV"].values
    var_f = dp_f.var(ddof=1)
    if var_f == 0:
        continue

    N_f = len(t_f)
    if N_f > N_MAX_SUBSAMPLE:
        idx_f = np.sort(rng.choice(N_f, N_MAX_SUBSAMPLE, replace=False))
        t_f   = t_f[idx_f]
        dp_f  = dp_f[idx_f]

    i_f, j_f = np.triu_indices(len(t_f), k=1)
    dt_f_all  = t_f[j_f]  - t_f[i_f]
    prod_f_all = dp_f[i_f] * dp_f[j_f]

    # restrict to 10-hour window
    ok_f   = (dt_f_all >= LAG_MIN_MIN) & (dt_f_all <= LAG_MAX_MIN)
    dt_f   = dt_f_all[ok_f]
    prod_f = prod_f_all[ok_f]

    bidx_f = np.digitize(dt_f, bin_edges) - 1
    val_f  = (bidx_f >= 0) & (bidx_f < N_BINS_LOG)

    c_f  = np.full(N_BINS_LOG, np.nan)
    ce_f = np.full(N_BINS_LOG, np.nan)
    np_f = np.zeros(N_BINS_LOG, dtype=int)

    for k in range(N_BINS_LOG):
        m  = val_f & (bidx_f == k)
        nk = m.sum()
        np_f[k] = nk
        if nk < MIN_PAIRS:
            continue
        pk = prod_f[m]
        c_f[k]  = pk.mean() / var_f
        ce_f[k] = pk.std(ddof=1) / (np.sqrt(nk) * var_f)

    corr_by_filter[filt]     = c_f
    corr_err_by_filter[filt] = ce_f
    npairs_by_filter[filt]   = np_f
    print(f"  {filt}: {len(sub)} obs, {len(t_f)} intra-night pairs used")

print("Done.")

In [ ]:
fig4, axes4 = plt.subplots(1, 2, figsize=(16, 6))
fig4.subplots_adjust(wspace=0.32)

for ax4, xdata, xlabel, xscale, xlim in [
    (axes4[0], lag_hr,  r"$\Delta t$ [hours]",  "linear",
     (0, LAG_MAX_MIN / 60.0)),
    (axes4[1], lag_min, r"$\Delta t$ [minutes]", "log",
     (LAG_MIN_MIN, LAG_MAX_MIN)),
]:
    ax4.plot(xdata[good], corr[good],
             color="black", lw=1.0, alpha=0.4, ls="--",
             label="All filters")

    for filt in FILTERS_OF_INTEREST:
        if filt not in corr_by_filter:
            continue
        c_f  = corr_by_filter[filt]
        ce_f = corr_err_by_filter[filt]
        gf   = npairs_by_filter[filt] >= MIN_PAIRS
        ax4.fill_between(
            xdata[gf], c_f[gf] - ce_f[gf], c_f[gf] + ce_f[gf],
            alpha=0.15, color=filter_color[filt]
        )
        ax4.plot(
            xdata[gf], c_f[gf],
            color=filter_color[filt], lw=1.8, marker="o", ms=3,
            label=filter_label[filt]
        )

    ax4.axhline(0.0,      ls="--", color=ZERO_COLOR,  lw=1.0)
    ax4.axhline(1.0/np.e, ls=":",  color=INV_E_COLOR, lw=1.5,
                label=r"$C = 1/e$")
    ax4.set_xscale(xscale)
    ax4.set_xlim(xlim)
    ax4.set_ylim(-0.35, 1.05)
    ax4.set_xlabel(xlabel, fontsize=13)
    ax4.set_ylabel(r"$C(\Delta t)$", fontsize=13)
    ax4.legend(fontsize=9, loc="upper right")

axes4[0].set_title("Per-filter correlation — linear hours", fontsize=11)
axes4[1].set_title("Per-filter correlation — log minutes",  fontsize=11)

fig4.suptitle(
    f"PWV two-point correlation by filter (0–10 h) — {version_run} (tight cuts)",
    fontsize=12, y=1.01
)

figfile4 = f"{pathfigs}/{prefix}_{version_run}_2point_corr_per_filter{figtype}"
fig4.savefig(figfile4, bbox_inches="tight")
print(f"Saved: {figfile4}")
plt.show()

---
## 10. Exponential fit of the decorrelation timescale

Fit $C(\Delta t) = A \exp(-\Delta t / \tau)$ where $C > 0$
to estimate $\tau$ within the 10-hour window.

In [ ]:
from scipy.optimize import curve_fit


def exp_decay(dt, A, tau):
    """Simple exponential decay model."""
    return A * np.exp(-dt / tau)


# Restrict fit to bins with C > 0, within [LAG_MIN_MIN, LAG_MAX_MIN]
fit_mask = (good
            & (lag_min <= LAG_MAX_MIN)
            & (corr > 0)
            & np.isfinite(corr)
            & np.isfinite(corr_err)
            & (corr_err > 0))

x_fit = lag_min[fit_mask]
y_fit = corr[fit_mask]
e_fit = corr_err[fit_mask]

tau_guess = tau_min if np.isfinite(tau_min) else 60.0

try:
    popt, pcov = curve_fit(
        exp_decay, x_fit, y_fit,
        p0=[1.0, tau_guess],
        sigma=e_fit, absolute_sigma=True,
        bounds=([0, 0.1], [2.0, LAG_MAX_MIN]),   # tau bounded within 10-h window
        maxfev=20_000
    )
    perr = np.sqrt(np.diag(pcov))
    A_fit, tau_fit_min = popt
    A_err, tau_fit_err = perr
    tau_fit_hr = tau_fit_min / 60.0
    print("=== Exponential fit  C(dt) = A * exp(-dt/tau) ===")
    print(f"  A   = {A_fit:.4f} +/- {A_err:.4f}")
    print(f"  tau = {tau_fit_min:.1f} +/- {tau_fit_err:.1f} min")
    print(f"      = {tau_fit_hr:.2f} +/- {tau_fit_err/60:.2f} h")
    fit_ok = True
except Exception as exc:
    print(f"Fit failed: {exc}")
    fit_ok = False

In [ ]:
fig5, axes5 = plt.subplots(1, 2, figsize=(16, 6))
fig5.subplots_adjust(wspace=0.32)

if fit_ok:
    t_dense_min = np.logspace(
        np.log10(LAG_MIN_MIN), np.log10(LAG_MAX_MIN), 500
    )
    t_dense_hr = t_dense_min / 60.0
    c_dense    = exp_decay(t_dense_min, *popt)

for ax5, xdata, x_dense, xlabel, xscale, xlim, x_tau in [
    (axes5[0], lag_hr,  t_dense_hr  if fit_ok else None,
     r"$\Delta t$ [hours]",   "linear",
     (0, LAG_MAX_MIN / 60.0),
     tau_fit_hr  if fit_ok else np.nan),
    (axes5[1], lag_min, t_dense_min if fit_ok else None,
     r"$\Delta t$ [minutes]", "log",
     (LAG_MIN_MIN, LAG_MAX_MIN),
     tau_fit_min if fit_ok else np.nan),
]:
    ax5.fill_between(
        xdata[good], corr[good] - corr_err[good], corr[good] + corr_err[good],
        alpha=0.25, color=CORR_COLOR
    )
    ax5.plot(xdata[good], corr[good],
             color=CORR_COLOR, lw=1.5, marker="o", ms=3.5, zorder=3,
             label=r"$C(\Delta t)$")

    if fit_ok:
        ax5.plot(x_dense, c_dense,
                 color="darkred", lw=2.0, ls="-", zorder=5,
                 label=(rf"$A\,e^{{-\Delta t/\tau}}$"
                        rf", $\tau={tau_fit_hr:.1f}$ h"))
        unit = xlabel.split('[')[1].rstrip(']') if '[' in xlabel else ''
        ax5.axvline(x_tau, ls="--", color=TAU_COLOR, lw=1.5,
                    label=rf"$\tau = {x_tau:.1f}$ {unit}")

    ax5.axhline(0.0,      ls="--", color=ZERO_COLOR,  lw=1.0)
    ax5.axhline(1.0/np.e, ls=":",  color=INV_E_COLOR, lw=1.5,
                label=r"$C = 1/e$")
    ax5.set_xscale(xscale)
    ax5.set_xlim(xlim)
    ax5.set_ylim(-0.25, 1.05)
    ax5.set_xlabel(xlabel, fontsize=13)
    ax5.set_ylabel(r"$C(\Delta t)$", fontsize=13)
    ax5.legend(fontsize=9, loc="upper right")

axes5[0].set_title("Exponential fit — linear hours", fontsize=11)
axes5[1].set_title("Exponential fit — log minutes",  fontsize=11)

fig5.suptitle(
    f"PWV correlation + exponential fit (0–10 h) — {version_run} (tight cuts)",
    fontsize=12, y=1.01
)

figfile5 = f"{pathfigs}/{prefix}_{version_run}_2point_corr_expfit{figtype}"
fig5.savefig(figfile5, bbox_inches="tight")
print(f"Saved: {figfile5}")
plt.show()

---
## 11. Summary & export

In [ ]:
print("=== PWV two-point temporal correlation summary ===")
print(f"  N observations (after cuts) : {len(df_keep)}")
print(f"  Sub-sample used             : {len(t_s)}")
print(f"  Pairs within 10-h window    : {len(dt_pairs):,}")
print(f"  Lag bins (log-spaced)       : {N_BINS_LOG}")
print(f"  Lag range                   : {LAG_MIN_MIN:.0f} min – {LAG_MAX_MIN:.0f} min ({LAG_MAX_MIN/60:.0f} h)")
print()
if np.isfinite(tau_min):
    print(f"  1/e crossing (empirical)    : {tau_min:.1f} min = {tau_hr:.2f} h")
else:
    print(f"  C(Delta_t) does not drop below 1/e within 10 h")
if fit_ok:
    print(f"  Exp-fit tau                 : {tau_fit_min:.1f} +/- {tau_fit_err:.1f} min")
    print(f"                                {tau_fit_hr:.2f} +/- {tau_fit_err/60:.2f} h")

In [ ]:
# Save correlation array as CSV
df_corr_out = pd.DataFrame({
    "lag_min" : lag_min,
    "lag_hr"  : lag_hr,
    "C"       : corr,
    "C_err"   : corr_err,
    "n_pairs" : n_pairs,
})
csv_file = f"{pathfigs}/{prefix}_{version_run}_2point_corr.csv"
df_corr_out.to_csv(csv_file, index=False, float_format="%.6f")
print(f"Saved: {csv_file}")

# Save scalar results as JSON
results = {
    "version_run"     : version_run,
    "lag_max_min"     : LAG_MAX_MIN,
    "lag_max_hr"      : LAG_MAX_MIN / 60.0,
    "N_obs"           : int(len(df_keep)),
    "N_subsample"     : int(len(t_s)),
    "N_pairs_10h"     : int(len(dt_pairs)),
    "tau_1e_min"      : float(tau_min) if np.isfinite(tau_min) else None,
    "tau_1e_hr"       : float(tau_hr)  if np.isfinite(tau_hr)  else None,
}
if fit_ok:
    results.update({
        "exp_A"          : float(A_fit),
        "exp_A_err"      : float(A_err),
        "exp_tau_min"    : float(tau_fit_min),
        "exp_tau_err_min": float(tau_fit_err),
        "exp_tau_hr"     : float(tau_fit_hr),
    })

json_file = f"{pathfigs}/{prefix}_{version_run}_2point_corr_results.json"
with open(json_file, "w") as fh:
    json.dump(results, fh, indent=2)
print(f"Saved: {json_file}")
print(json.dumps(results, indent=2))